# Correlation Tutorial 1: Getting Started with Structural Analysis

Welcome to **Correlation**, a high-performance C++23 atomic structural analysis suite with zero-overhead Python bindings.

This tutorial guides you through:
1. Creating or importing atomic cell configurations.
2. Computing the **Radial Distribution Function (RDF) $g(r)$**.
3. Computing the **Plane / Bond Angle Distribution (PAD / BAD)**.
4. Extracting total and element-resolved partial distributions.
5. Visualizing distributions with Matplotlib.


## 1. Imports and Verification


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import correlation

print("Correlation version:", getattr(correlation, "__version__", "4.0.0"))
print("Available calculators:", correlation.get_registered_calculators())


## 2. Building an Atomic Cell

You can instantiate a `correlation.Cell` directly by specifying lattice parameters (lengths $a, b, c$ and angles $\alpha, \beta, \gamma$) and adding atomic positions.


In [ ]:
# Create a cubic unit cell (e.g. Si fcc diamond lattice parameter a = 5.43 Å)
cell = correlation.Cell(5.43, 5.43, 5.43, 90.0, 90.0, 90.0)

# Add Silicon atoms (fractional diamond coordinates)
si_frac = [
    [0.0, 0.0, 0.0], [0.5, 0.5, 0.0], [0.5, 0.0, 0.5], [0.0, 0.5, 0.5],
    [0.25, 0.25, 0.25], [0.75, 0.75, 0.25], [0.75, 0.25, 0.75], [0.25, 0.75, 0.75]
]

# Convert fractional coordinates to Cartesian and add to cell
for frac in si_frac:
    pos = [frac[0] * 5.43, frac[1] * 5.43, frac[2] * 5.43]
    cell.add_atom(correlation.Atom("Si", pos[0], pos[1], pos[2]))

print(f"Cell created with {cell.atom_count()} atoms and volume {cell.volume:.2f} Å³.")


## 3. Running Distribution Function Analysis

The `DistributionFunctions` engine coordinates structural calculators. We initialize it with the cell, a maximum cutoff radius, and optional bond cutoffs.


In [ ]:
# Initialize DistributionFunctions container for the cell
df = correlation.DistributionFunctions(cell)

# Calculate Radial Distribution Function g(r) up to r_max = 10.0 Å with 0.05 Å bin width
rdf_params = correlation.RDFParams(r_max=10.0, r_bin_width=0.05)
df.calculate_rdf(rdf_params)

# Calculate Plane Angle Distribution (PAD) with 0.5 degree bin width
df.calculate_pad(0.5)

print("Available histograms:", df.get_available_histograms())


## 4. Extracting Histogram Data & Partials

Histograms contain:
- `bins`: Radial distances (Å) or angles (degrees).
- `partials`: Dictionary mapping pair / triplet keys (e.g. `'Si-Si'`, `'Total'`) to density values.


In [ ]:
# Extract g(r)
gr_hist = df.get_histogram("g_r")
r_bins = np.array(gr_hist.bins)
print(f"Radial bins: {len(r_bins)} points, range [{r_bins[0]:.2f}, {r_bins[-1]:.2f}] Å")

for key in gr_hist.partials:
    values = np.array(gr_hist.partials[key])
    print(f"  Partial: {key} (length {len(values)})")


## 5. Visualizing the Distributions

Let's plot $g(r)$ and the angular distribution using Matplotlib.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), dpi=120)

# Plot g(r)
for key, vals in gr_hist.partials.items():
    ax1.plot(gr_hist.bins, vals, label=key, lw=1.8)
ax1.set_xlabel("Radius r (Å)", fontsize=11)
ax1.set_ylabel("g(r)", fontsize=11)
ax1.set_title("Radial Distribution Function (RDF)", fontsize=12, fontweight='bold')
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend()

# Plot PAD if available
if "PAD" in df.get_available_histograms():
    pad_hist = df.get_histogram("PAD")
    for key, vals in pad_hist.partials.items():
        ax2.plot(pad_hist.bins, vals, label=key, lw=1.8, color="#009E73")
    ax2.set_xlabel("Bond Angle θ (deg)", fontsize=11)
    ax2.set_ylabel("P(θ)", fontsize=11)
    ax2.set_title("Plane Angle Distribution (PAD)", fontsize=12, fontweight='bold')
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend()

plt.tight_layout()
plt.show()


## Conclusion
You have successfully constructed a structure, performed high-speed structural calculations, and visualized the distribution functions.

Next tutorial: `02_ase_pymatgen_interop.ipynb` shows how to seamlessly integrate Correlation into existing workflows with ASE and Pymatgen.
